# The model boundary

Our app sends a prompt. The model sends a response back. The point where they meet is the **model boundary**.

In this notebook, we will learn how to:

- find the answer inside a model response;
- ask for JSON instead of loose text;
- read JSON even when the model adds extra text;
- stop when the reply is unusable;
- record tokens and cost; and
- tell the difference between valid JSON and a correct answer.

## Setup

Run this cell first. It loads the same model settings used by the course.

In [ ]:
import json
import os
import time
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent

load_dotenv(course_root / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.getenv("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")
TEMPERATURE = 0

litellm.drop_params = True
litellm.cache = litellm.Cache(
    type="disk",
    disk_cache_dir=str(course_root / ".litellm_cache"),
)

## 1. Make one model call

We send a list of messages. Each message has a role and some content.

In [ ]:
response = litellm.completion(
    model=MODEL,
    temperature=TEMPERATURE,
    messages=[
        {
            "role": "user",
            "content": "What is a restocking fee? Use one short sentence.",
        }
    ],
)

print(response.choices[0].message.content)

The answer is inside response.choices[0].message.content.

The response also contains useful facts about the call.

In [ ]:
choice = response.choices[0]

print("role:", choice.message.role)
print("finish reason:", choice.finish_reason)
print("prompt tokens:", response.usage.prompt_tokens)
print("answer tokens:", response.usage.completion_tokens)
print("cost in USD:", response._hidden_params.get("response_cost"))

A finish reason of stop means the model finished normally. A finish reason of length means it ran out of allowed tokens.

The API does not remember a conversation for us. Each call must include the messages the model needs.

## 2. See why loose text is hard to use

We will use one return ticket. The gold field is the known correct answer. We hide it from the model.

In [ ]:
from shoplab import world

ticket = world.load_tickets()["train"][1]
ticket_for_model = {
    key: value
    for key, value in ticket.items()
    if key != "gold"
}

ticket_for_model

In [ ]:
plain_prompt = (
    "You handle returns for an outdoor shop. "
    "Read this ticket and decide what to do.\n\n"
    + json.dumps(ticket_for_model, indent=2)
)

plain_response = litellm.completion(
    model=MODEL,
    temperature=TEMPERATURE,
    max_tokens=180,
    messages=[{"role": "user", "content": plain_prompt}],
)

print(plain_response.choices[0].message.content)

A person can read this answer. A program cannot safely find a decision, policy, and refund amount inside changing prose.

Let us ask for clear fields.

## 3. Ask for JSON

We show the model the exact shape we want.

In [ ]:
shape = {
    "decision": "short label",
    "policy_id": "policy id",
    "refund_usd": "number or null",
}

json_prompt = (
    plain_prompt
    + "\n\nReturn one JSON object with this shape. Add no other text:\n"
    + json.dumps(shape)
)

json_text = litellm.completion(
    model=MODEL,
    temperature=TEMPERATURE,
    max_tokens=180,
    messages=[{"role": "user", "content": json_prompt}],
).choices[0].message.content

print(json_text)

json.loads works when the whole reply is valid JSON.

In [ ]:
try:
    value = json.loads(json_text)
    print("parsed:", value)
except json.JSONDecodeError as error:
    print("could not parse the whole reply:", error)

## 4. Handle extra text

Models sometimes put JSON inside a Markdown fence or add a sentence around it. The JSON may be valid even though the whole reply is not.

In [ ]:
fence = chr(96) * 3
wrapped_reply = (
    "Here is the result:\n"
    + fence
    + 'json\n{"decision": "approve_refund"}\n'
    + fence
)

try:
    json.loads(wrapped_reply)
except json.JSONDecodeError as error:
    print("json.loads failed:", error)

We need a small helper.

It looks for an opening brace or bracket. At each one, it asks Python's JSON decoder to try. The decoder handles nesting and quoted text for us.

In [ ]:
def parse_model_json(text: str):
    decoder = json.JSONDecoder()

    for start, character in enumerate(text):
        if character not in "[{":
            continue

        try:
            value, _ = decoder.raw_decode(text[start:])
            return value
        except json.JSONDecodeError:
            pass

    raise ValueError("no valid JSON found in model output")

In [ ]:
examples = [
    '{"decision": "deny"}',
    fence + 'json\n{"decision": "deny"}\n' + fence,
    'My answer: {"decision": "approve_refund"} Thank you.',
    "Approve the refund.",
]

for example in examples:
    try:
        print("parsed:", parse_model_json(example))
    except ValueError as error:
        print("stopped:", error)

The last reply has no JSON, so the helper raises ValueError.

That is safer than returning None or an empty object. A bad reply should stop here, where we can see the real problem.

## 5. Record every call

One call is cheap. Many calls can be slow and expensive.

We can make one function that calls the model and records a receipt.

In [ ]:
CALLS = []

def ask(prompt: str, max_tokens=200) -> str:
    started = time.perf_counter()

    result = litellm.completion(
        model=MODEL,
        temperature=TEMPERATURE,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )

    CALLS.append({
        "prompt_tokens": result.usage.prompt_tokens,
        "answer_tokens": result.usage.completion_tokens,
        "cost_usd": result._hidden_params.get("response_cost") or 0,
        "seconds": round(time.perf_counter() - started, 3),
    })

    return result.choices[0].message.content

def total_usage():
    return {
        "calls": len(CALLS),
        "prompt_tokens": sum(call["prompt_tokens"] for call in CALLS),
        "answer_tokens": sum(call["answer_tokens"] for call in CALLS),
        "cost_usd": sum(call["cost_usd"] for call in CALLS),
    }

In [ ]:
print(ask("Reply with OK.", max_tokens=10))
print("last call:", CALLS[-1])
print("total:", total_usage())

The course already has this idea in shoplab.llm.

- complete makes a call and adds a row to LEDGER.
- llm is a short helper that returns only the answer text.
- usage_summary adds the rows together.

Later notebooks use these helpers instead of calling LiteLLM directly.

In [ ]:
from shoplab.llm import LEDGER, complete, llm, usage_summary

print(llm("Reply with READY.", max_tokens=10))
print("last ledger row:", LEDGER[-1])
print("ledger total:", usage_summary())

## 6. Valid JSON can still be wrong

Our ticket contains a product code. It does not contain the product price or the shop's return policy.

The model needs those facts to calculate the real answer. We will ask anyway and compare its guess with the known answer.

In [ ]:
final_prompt = (
    "Decide this return request.\n\n"
    + json.dumps(ticket_for_model, indent=2)
    + "\n\nReturn only JSON with decision, policy_id, and refund_usd."
)

final_text = llm(final_prompt, max_tokens=180)
try:
    model_answer = parse_model_json(final_text)
except ValueError as error:
    model_answer = f"The parser stopped: {error}"

print("model:", model_answer)
print("known:", ticket["gold"])

The model may return perfect JSON and still get the policy or amount wrong.

The parser fixed the format. It could not add missing facts. A model without the needed data can only guess.

This is the main lesson: **valid JSON is not the same as a valid decision**.

## A note about temperature 0

Temperature 0 makes answers less varied. It does not promise the exact same answer forever. Providers can update models and serving systems.

The disk cache gives us exact repeats for requests it has already saved.

## What to remember

1. The model's text is inside a larger response object.
2. Ask for the output shape your program needs.
3. Expect fences and extra text around JSON.
4. Raise an error when there is no usable JSON.
5. Send every call through one function that records usage.
6. Good structure does not make unsupported facts true.

Next, we will give the model tools so it can look up facts instead of guessing.

## Small practice steps

1. Add a confidence field to shape. Does the model return it?
2. Give parse_model_json two JSON objects. Which one does it return?
3. Run the final prompt with STRONG_MODEL. Can a stronger model know a price it was never given?